In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Dataset
os.system("kaggle datasets download -d asdasdasasdas/garbage-classification")
os.system("unzip -q garbage-classification.zip -d garbage_data")

DATASET_PATH = "garbage_data/Garbage classification/Garbage classification/"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(DATASET_PATH,
    target_size=(224,224), batch_size=32, class_mode='categorical',
    subset='training', seed=42)

val_gen = train_datagen.flow_from_directory(DATASET_PATH,
    target_size=(224,224), batch_size=32, class_mode='categorical',
    subset='validation', seed=42)

# MobileNetV2
base = applications.MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base.trainable = False

inputs = layers.Input(shape=(224,224,3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(6, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    # DRIVE MEIN SAVE HOGA!
    ModelCheckpoint('/content/drive/MyDrive/WasteProject/best_model.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("Training shuru!")
history = model.fit(train_gen, epochs=10,
    validation_data=val_gen, callbacks=callbacks, verbose=1)

Found 2024 images belonging to 6 classes.
Found 503 images belonging to 6 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training shuru!
Epoch 1/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4986 - loss: 1.2944
Epoch 1: val_accuracy improved from None to 0.71571, saving model to /content/drive/MyDrive/WasteProject/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/WasteProject/best_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 158s 2s/step - accuracy: 0.6290 - loss: 0.9579 - val_accuracy: 0.7157 - val_loss: 0.7245
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8099 - loss: 0.5514
Epoch 2: val_accuracy improved from 0.71571 to 0.73161, saving model to /content/drive/MyDrive/WasteProject/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/WasteProject/best_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 155s 2s/step - accuracy: 0.8152 - loss: 0.5453 - val_accuracy: 0.7316 - val_loss: 0.7108
Epoch 3/10
64/64 ━━━━━━━━━━━━━━

In [ ]:
import os
print(os.path.exists('/content/drive/MyDrive/WasteProject/best_model.keras'))

True


In [ ]:
!pip install gradio -q
print("Gradio installed!")

Gradio installed!


In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.models import load_model
from PIL import Image

# Model load karo
model = load_model('/content/drive/MyDrive/WasteProject/best_model.keras')
classes = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

def predict_waste(image):
    img = Image.fromarray(image).resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    predictions = model.predict(img_array)
    confidences = {classes[i]: float(predictions[0][i]) for i in range(len(classes))}
    return confidences

# Web Interface
interface = gr.Interface(
    fn=predict_waste,
    inputs=gr.Image(),
    outputs=gr.Label(num_top_classes=3),
    title="🗑️ Waste Classification System",
    description="Upload a waste image to classify it into: Cardboard, Glass, Metal, Paper, Plastic, or Trash",
    examples=[]
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59ba31d7e5df71790a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
import numpy as np
import gradio as gr
from tensorflow.keras.models import load_model
from PIL import Image
from google.colab import drive

# Drive mount
drive.mount('/content/drive')

# Model load
model = load_model('/content/drive/MyDrive/WasteProject/best_model.keras')
classes = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

def predict_waste(image):
    img = Image.fromarray(image).resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    predictions = model.predict(img_array)
    confidences = {classes[i]: float(predictions[0][i]) for i in range(len(classes))}
    return confidences

interface = gr.Interface(
    fn=predict_waste,
    inputs=gr.Image(),
    outputs=gr.Label(num_top_classes=3),
    title="🗑️ Waste Classification System",
    description="Upload a waste image to classify it!",
)

interface.launch(share=True)

Mounted at /content/drive
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eff47c6635c552aba6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
recycling_tips = {
    'cardboard': '♻️ Recyclable! Fold karke recycle bin mein daalo.',
    'glass': '♻️ Recyclable! Glass bin mein daalo.',
    'metal': '♻️ Recyclable! Metal bin mein daalo.',
    'paper': '♻️ Recyclable! Paper bin mein daalo.',
    'plastic': '♻️ Recyclable! Plastic bin mein daalo.',
    'trash': '🚯 Non-recyclable. General waste bin mein daalo.'
}

def predict_waste_styled(image):
    img = Image.fromarray(image).resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    predictions = model.predict(img_array)
    confidences = {classes[i]: float(predictions[0][i]) for i in range(len(classes))}
    top_class = max(confidences, key=confidences.get)
    tip = recycling_tips[top_class]
    return confidences, tip

with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classification") as demo:
    gr.Markdown("# 🌍 Automated Waste Classification System")
    gr.Markdown("### Galgotias University | Deep Learning Project")
    gr.Markdown("---")

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(label="📸 Upload Waste Image")
            submit_btn = gr.Button("🔍 Classify!", variant="primary")

        with gr.Column():
            output_label = gr.Label(num_top_classes=6, label="📊 Classification Results")
            tip_output = gr.Textbox(label="♻️ Recycling Tip", interactive=False)

    submit_btn.click(
        fn=predict_waste_styled,
        inputs=image_input,
        outputs=[output_label, tip_output]
    )

demo.launch(share=True)

/tmp/ipykernel_6859/3106792501.py:20: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classification") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://489dc55964d97231d1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import random

recycling_tips = {
    'cardboard': ('♻️ Recyclable!', 'Fold it and put it in the recycling bin.', '#4CAF50'),
    'glass': ('♻️ Recyclable!', 'Put it in the glass bin. Rinse it first!', '#4CAF50'),
    'metal': ('♻️ Recyclable!', 'Put it in the metal bin.', '#4CAF50'),
    'paper': ('♻️ Recyclable!', 'Put it in the paper bin. Keep it dry!', '#4CAF50'),
    'plastic': ('♻️ Recyclable!', 'Put it in the plastic bin.', '#FF9800'),
    'trash': ('🚯 Non-Recyclable!', 'Put it in the general waste bin.', '#F44336')
}

co2_savings = {
    'cardboard': 0.9, 'glass': 0.3, 'metal': 1.5,
    'paper': 0.7, 'plastic': 0.4, 'trash': 0.0
}

fun_facts = {
    'cardboard': '📦 Fun Fact: Recycling 1 ton of cardboard saves 17 trees!',
    'glass': '🫙 Fun Fact: Glass takes around 1 million years to decompose!',
    'metal': '🥫 Fun Fact: Aluminium can be recycled infinitely without losing quality!',
    'paper': '📄 Fun Fact: Recycling 1 ton of paper saves about 26,000 liters of water!',
    'plastic': '🧴 Fun Fact: Plastic takes up to 500 years to decompose!',
    'trash': '🗑️ Fun Fact: Proper waste management can reduce greenhouse gases by up to 20%!'
}

classify_count = [0]
total_co2 = [0.0]

def predict_waste_full(image):
    img = Image.fromarray(image).resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    predictions = model.predict(img_array)
    confidences = {classes[i]: float(predictions[0][i]) for i in range(len(classes))}

    top_class = max(confidences, key=confidences.get)
    top_conf = confidences[top_class] * 100

    status, tip, color = recycling_tips[top_class]
    co2 = co2_savings[top_class]
    fact = fun_facts[top_class]

    classify_count[0] += 1
    total_co2[0] += co2

    result = f"""
## {status}
**Category:** {top_class.upper()} ({top_conf:.1f}% confidence)

**Tip:** {tip}

{fact}

---
🌍 **CO2 Saved this item:** {co2} kg
📊 **Total items classified:** {classify_count[0]}
💚 **Total CO2 saved today:** {total_co2[0]:.2f} kg
    """

    return confidences, result

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Waste Classification System",
    css="""
    .gradio-container {
        font-family: 'Arial', sans-serif;
    }
    .main-header {
        text-align: center;
        padding: 20px;
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        border-radius: 15px;
        color: white;
        margin-bottom: 20px;
    }
    """
) as demo:

    gr.HTML("""
    <div class="main-header">
        <h1>🌍 Automated Waste Classification System</h1>
        <p>Galgotias University | CNN-BiLSTM Deep Learning Project</p>
        <p>Khushi Dubey & Suhani</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(label="📸 Upload Waste Image")
            classify_btn = gr.Button("🔍 Classify Waste!", variant="primary", size="lg")
            gr.Markdown("### 📂 Try these categories:")
            gr.Markdown("Glass • Paper • Cardboard • Plastic • Metal • Trash")

        with gr.Column(scale=1):
            output_label = gr.Label(num_top_classes=6, label="📊 Confidence Scores")
            result_output = gr.Markdown(label="Results")

    classify_btn.click(
        fn=predict_waste_full,
        inputs=image_input,
        outputs=[output_label, result_output]
    )

    gr.HTML("""
    <div style='text-align:center; padding:15px; color:gray;'>
        <p>🌱 Every correct classification helps our environment!</p>
    </div>
    """)

demo.launch(share=True)

/tmp/ipykernel_6859/311151240.py:62: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_6859/311151240.py:62: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fe0268e938f35430aa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo.close()

NameError: name 'demo' is not defined

In [ ]:
"""
Waste Classification System — Upgraded Version
Khushi Dubey & Suhani | Galgotias University
New Features: Webcam Support + Green/RED Color Coding + Recycling Impact Score
"""

import gradio as gr
import numpy as np
from PIL import Image

# ─────────────────────────────────────────────
# Data maps
# ─────────────────────────────────────────────

recycling_tips = {
    'cardboard': ('♻️ Recyclable',  'Flatten it and drop in the recycling bin.',  'recyclable'),
    'glass':     ('♻️ Recyclable',  'Rinse and place in the glass bin.',           'recyclable'),
    'metal':     ('♻️ Recyclable',  'Clean and put in the metal / mixed bin.',     'recyclable'),
    'paper':     ('♻️ Recyclable',  'Keep it dry and put in the paper bin.',       'recyclable'),
    'plastic':   ('⚠️ Check Label', 'Check recycling number — some plastics are accepted.', 'caution'),
    'trash':     ('🚯 Non-Recyclable', 'Place in the general waste / landfill bin.', 'nonrecyclable'),
}

co2_savings = {
    'cardboard': 0.9, 'glass': 0.3, 'metal': 1.5,
    'paper': 0.7,     'plastic': 0.4, 'trash': 0.0,
}

impact_scores = {          # out of 10
    'cardboard': 8, 'glass': 7, 'metal': 9,
    'paper': 8,     'plastic': 5, 'trash': 0,
}

fun_facts = {
    'cardboard': '📦 Recycling 1 ton of cardboard saves 17 trees!',
    'glass':     '🫙 Glass takes ~1 million years to decompose in landfill.',
    'metal':     '🥫 Aluminium can be recycled infinitely without quality loss!',
    'paper':     '📄 Recycling 1 ton of paper saves ~26,000 litres of water.',
    'plastic':   '🧴 Plastic takes up to 500 years to decompose.',
    'trash':     '🗑️ Proper waste management can cut greenhouse gases by 20%.',
}

# Running totals
classify_count = [0]
total_co2      = [0.0]

classes = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

# ─────────────────────────────────────────────
# Core prediction function
# ─────────────────────────────────────────────

def predict_waste(image):
    """
    image: numpy array from Gradio (webcam or upload).
    Returns (label_dict, html_result_string).
    """
    if image is None:
        return {}, "<p style='color:gray;'>No image provided.</p>"

    img = Image.fromarray(image).resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # ── Replace the line below with your real model.predict() ──
    predictions = model.predict(img_array)          # shape: (1, 6)
    # ───────────────────────────────────────────────────────────

    confidences  = {classes[i]: float(predictions[0][i]) for i in range(len(classes))}
    top_class    = max(confidences, key=confidences.get)
    top_conf     = confidences[top_class] * 100

    status, tip, category = recycling_tips[top_class]
    co2          = co2_savings[top_class]
    score        = impact_scores[top_class]
    fact         = fun_facts[top_class]

    classify_count[0] += 1
    total_co2[0]      += co2

    # ── Color coding ─────────────────────────
    if category == 'recyclable':
        badge_color = '#1B5E20'   # dark green
        bg_color    = '#E8F5E9'   # light green
        border      = '#4CAF50'
        emoji       = '🟢'
    elif category == 'caution':
        badge_color = '#E65100'   # dark orange
        bg_color    = '#FFF3E0'
        border      = '#FF9800'
        emoji       = '🟡'
    else:                          # non-recyclable
        badge_color = '#B71C1C'   # dark red
        bg_color    = '#FFEBEE'
        border      = '#F44336'
        emoji       = '🔴'

    # ── Impact score bar ─────────────────────
    filled_blocks = score
    empty_blocks  = 10 - score
    score_bar     = '🟩' * filled_blocks + '⬜' * empty_blocks

    result_html = f"""
    <div style="
        background:{bg_color};
        border:3px solid {border};
        border-radius:16px;
        padding:20px;
        font-family:Arial,sans-serif;
        margin-top:10px;
    ">
        <div style="
            background:{badge_color};
            color:white;
            border-radius:10px;
            padding:10px 18px;
            font-size:1.4em;
            font-weight:bold;
            margin-bottom:14px;
            display:inline-block;
        ">{emoji} {status} — {top_class.upper()}</div>

        <p style="font-size:1.05em;color:#333;">
            <strong>Confidence:</strong> {top_conf:.1f}%
        </p>
        <p style="font-size:1.05em;color:#333;">
            <strong>♻️ Recycling Tip:</strong> {tip}
        </p>
        <p style="font-size:1.05em;color:#333;">
            <strong>🌱 CO₂ Saved:</strong> {co2} kg per item recycled
        </p>

        <hr style="border:1px solid {border};margin:12px 0;">

        <p style="font-size:1.05em;color:#333;">
            <strong>📊 Recycling Impact Score:</strong> {score}/10<br>
            {score_bar}
        </p>
        <p style="font-size:0.95em;color:#555;">{fact}</p>

        <hr style="border:1px solid {border};margin:12px 0;">

        <p style="color:#555;font-size:0.9em;">
            📦 Items classified today: <strong>{classify_count[0]}</strong> &nbsp;|&nbsp;
            💚 Total CO₂ saved: <strong>{total_co2[0]:.2f} kg</strong>
        </p>
    </div>
    """
    return confidences, result_html


# ─────────────────────────────────────────────
# Gradio UI
# ─────────────────────────────────────────────

custom_css = """
.gradio-container { font-family: Arial, sans-serif; }
.main-header {
    text-align:center;
    padding:22px;
    background:linear-gradient(135deg,#2E7D32 0%,#1565C0 100%);
    border-radius:16px;
    color:white;
    margin-bottom:20px;
}
.tab-label { font-size:1.1em; font-weight:bold; }
"""

with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classification System", css=custom_css) as demo:

    gr.HTML("""
    <div class="main-header">
        <h1>🌍 Automated Waste Classification System</h1>
        <p style="font-size:1.05em;">Galgotias University &nbsp;|&nbsp; CNN‑BiLSTM Deep Learning Project</p>
        <p>Khushi Dubey &amp; Suhani</p>
    </div>
    """)

    # ── Legend ───────────────────────────────
    gr.HTML("""
    <div style="display:flex;gap:14px;justify-content:center;margin-bottom:16px;flex-wrap:wrap;">
        <span style="background:#E8F5E9;border:2px solid #4CAF50;border-radius:8px;padding:6px 14px;">
            🟢 <strong>Recyclable</strong>
        </span>
        <span style="background:#FFF3E0;border:2px solid #FF9800;border-radius:8px;padding:6px 14px;">
            🟡 <strong>Check Label</strong>
        </span>
        <span style="background:#FFEBEE;border:2px solid #F44336;border-radius:8px;padding:6px 14px;">
            🔴 <strong>Non‑Recyclable</strong>
        </span>
    </div>
    """)

    # ── Tabs: Upload vs Webcam ────────────────
    with gr.Tabs():

        with gr.Tab("📁 Upload Image"):
            with gr.Row():
                with gr.Column(scale=1):
                    upload_input = gr.Image(label="Upload Waste Image", type="numpy")
                    upload_btn   = gr.Button("🔍 Classify", variant="primary", size="lg")
                with gr.Column(scale=1):
                    upload_label  = gr.Label(num_top_classes=6, label="Confidence Scores")
                    upload_result = gr.HTML(label="Result")

            upload_btn.click(
                fn=predict_waste,
                inputs=upload_input,
                outputs=[upload_label, upload_result],
            )

        with gr.Tab("📷 Webcam"):
            with gr.Row():
                with gr.Column(scale=1):
                    webcam_input = gr.Image(
                        label="Webcam — point at waste item",
                        sources=["webcam"],   # 👈 webcam fix
                        type="numpy",
                        streaming=False,
                    )
                    webcam_btn = gr.Button("🔍 Classify Snapshot", variant="primary", size="lg")
                with gr.Column(scale=1):
                    webcam_label  = gr.Label(num_top_classes=6, label="Confidence Scores")
                    webcam_result = gr.HTML(label="Result")

            webcam_btn.click(
                fn=predict_waste,
                inputs=webcam_input,
                outputs=[webcam_label, webcam_result],
            )

    gr.HTML("""
    <div style="text-align:center;padding:14px;color:gray;font-size:0.9em;">
        🌱 Every correct classification helps our planet!
    </div>
    """)

demo.launch(share=True)

/tmp/ipykernel_17892/3172367695.py:169: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classification System", css=custom_css) as demo:
/tmp/ipykernel_17892/3172367695.py:169: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classification System", css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed6c6d7f1f822eb9fa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
# Check karo model kahan hai
for root, dirs, files in os.walk('/content/drive/MyDrive/WasteProject'):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
# Step 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.system("pip install kaggle -q")
os.makedirs('/root/.kaggle', exist_ok=True)

# Upload your kaggle.json first, then run this
from google.colab import files
files.upload()  # select kaggle.json from your computer

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"khushidubey123","key":"9cac9acc6bcac9468b0e70b8ac49e5ca"}'}

In [ ]:
import os
os.system("mv kaggle.json /root/.kaggle/kaggle.json")
os.system("chmod 600 /root/.kaggle/kaggle.json")
os.system("kaggle datasets download -d asdasdasasdas/garbage-classification")
os.system("unzip -q garbage-classification.zip -d garbage_data")

# Verify
print(os.listdir("garbage_data/Garbage classification/Garbage classification/"))

['plastic', 'metal', 'glass', 'trash', 'cardboard', 'paper']


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

os.makedirs('/content/drive/MyDrive/WasteProject', exist_ok=True)

DATASET_PATH = "garbage_data/Garbage classification/Garbage classification/"

train_datagen = ImageDataGenerator(rescale=1./255, horizontal_flip=True,
                                   rotation_range=15, validation_split=0.2)

train_gen = train_datagen.flow_from_directory(DATASET_PATH,
    target_size=(224,224), batch_size=32, class_mode='categorical',
    subset='training', seed=42)

val_gen = train_datagen.flow_from_directory(DATASET_PATH,
    target_size=(224,224), batch_size=32, class_mode='categorical',
    subset='validation', seed=42)

base = applications.MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base.trainable = False

inputs = layers.Input(shape=(224,224,3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(6, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(train_gen, epochs=10, validation_data=val_gen, callbacks=[
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    ModelCheckpoint('/content/drive/MyDrive/WasteProject/best_model.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1)
], verbose=1)

# Save class names too
import json
with open('/content/drive/MyDrive/WasteProject/class_names.json', 'w') as f:
    json.dump(train_gen.class_indices, f)

print("Done! Files saved:")
print(os.listdir('/content/drive/MyDrive/WasteProject'))

Found 2024 images belonging to 6 classes.
Found 503 images belonging to 6 classes.
Epoch 1/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5009 - loss: 1.3396
Epoch 1: val_accuracy improved from None to 0.67793, saving model to /content/drive/MyDrive/WasteProject/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/WasteProject/best_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 159s 2s/step - accuracy: 0.6314 - loss: 0.9909 - val_accuracy: 0.6779 - val_loss: 0.7866
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7667 - loss: 0.6381
Epoch 2: val_accuracy improved from 0.67793 to 0.71173, saving model to /content/drive/MyDrive/WasteProject/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/WasteProject/best_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 143s 2s/step - accuracy: 0.7722 - loss: 0.6057 - val_accuracy: 0.7117 - val_loss: 0.7029
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8375 - loss: 0.4435
Epoch 3: val_

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/WasteProject'))

['best_model.keras', 'class_names.json']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
import json

model = tf.keras.models.load_model('/content/drive/MyDrive/WasteProject/best_model.keras')
with open('/content/drive/MyDrive/WasteProject/class_names.json', 'r') as f:
    class_indices = json.load(f)
classes = [k for k, v in sorted(class_indices.items(), key=lambda x: x[1])]
print("Model loaded! Classes:", classes)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model loaded! Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [ ]:
import gradio as gr
import numpy as np
from PIL import Image
from datetime import datetime

recycling_tips = {
    "cardboard": ("Recyclable",     "Flatten and place in the recycling bin.",   "recyclable"),
    "glass":     ("Recyclable",     "Rinse and place in the glass bin.",         "recyclable"),
    "metal":     ("Recyclable",     "Clean and place in the metal bin.",         "recyclable"),
    "paper":     ("Recyclable",     "Keep dry and place in the paper bin.",      "recyclable"),
    "plastic":   ("Check Label",    "Check recycling number on packaging.",      "caution"),
    "trash":     ("Non-Recyclable", "Place in the general waste bin.",           "nonrecyclable"),
}
co2_savings   = {"cardboard":0.9,"glass":0.3,"metal":1.5,"paper":0.7,"plastic":0.4,"trash":0.0}
impact_scores = {"cardboard":8,"glass":7,"metal":9,"paper":8,"plastic":5,"trash":0}
fun_facts = {
    "cardboard": "Recycling 1 ton of cardboard saves 17 trees!",
    "glass":     "Glass takes 1 million years to decompose in landfill.",
    "metal":     "Aluminium can be recycled infinitely without quality loss!",
    "paper":     "Recycling 1 ton of paper saves 26,000 litres of water.",
    "plastic":   "Plastic takes up to 500 years to decompose.",
    "trash":     "Proper waste management can cut greenhouse gases by 20%.",
}
bin_locations = {
    "recyclable":    [{"name":"Blue Recycling Bin - Sector 1","lat":28.4744,"lon":77.5040},
                      {"name":"Blue Recycling Bin - Sector 4","lat":28.4780,"lon":77.5060},
                      {"name":"Community Recycle Center","lat":28.4720,"lon":77.5020}],
    "caution":       [{"name":"Plastic Collection Point","lat":28.4744,"lon":77.5040},
                      {"name":"Plastic Recycling Drop-off","lat":28.4760,"lon":77.5070}],
    "nonrecyclable": [{"name":"General Waste Bin - Block A","lat":28.4730,"lon":77.5030},
                      {"name":"General Waste Bin - Block C","lat":28.4755,"lon":77.5055},
                      {"name":"Municipal Waste Collection","lat":28.4710,"lon":77.5010}],
}

history_log    = []
classify_count = [0]
total_co2      = [0.0]

def get_history_html():
    if not history_log:
        return "<p style='color:#333;text-align:center;padding:20px;'>No classifications yet.</p>"
    rows = ""
    for i, e in enumerate(reversed(history_log[-10:])):
        bg = "#f0f0f0" if i % 2 == 0 else "#ffffff"
        rows += (
            "<tr style='background:" + bg + ";'>"
            "<td style='padding:10px;color:#111;text-align:center;'>" + e["time"] + "</td>"
            "<td style='padding:10px;color:#111;text-align:center;font-weight:bold;'>" + e["item"] + "</td>"
            "<td style='padding:10px;color:#111;text-align:center;'>" + e["confidence"] + "</td>"
            "<td style='padding:10px;color:#111;text-align:center;'>" + e["emoji"] + " " + e["status"] + "</td>"
            "<td style='padding:10px;color:#111;text-align:center;'>" + str(e["co2"]) + " kg</td>"
            "</tr>"
        )
    return (
        "<div style='font-family:Arial;padding:10px;'>"
        "<h3 style='color:#2E7D32;'>Classification History (Last 10)</h3>"
        "<table style='width:100%;border-collapse:collapse;border:1px solid #ddd;'>"
        "<thead><tr style='background:#2E7D32;color:white;'>"
        "<th style='padding:10px;'>Time</th><th style='padding:10px;'>Item</th>"
        "<th style='padding:10px;'>Confidence</th><th style='padding:10px;'>Status</th>"
        "<th style='padding:10px;'>CO2 Saved</th></tr></thead>"
        "<tbody>" + rows + "</tbody></table>"
        "<p style='color:#333;margin-top:10px;'>Total CO2 saved: <strong>" + str(round(total_co2[0],2)) + " kg</strong></p>"
        "</div>"
    )

def get_map_html(category):
    locs  = bin_locations.get(category, bin_locations["nonrecyclable"])
    label = "Recycling Bins" if category=="recyclable" else "Plastic Drop-off Points" if category=="caution" else "General Waste Bins"
    color = "#1B5E20" if category=="recyclable" else "#E65100" if category=="caution" else "#B71C1C"
    markers = ""
    for loc in locs:
        markers += (
            "<div style='background:white;border:2px solid #4CAF50;border-radius:10px;padding:12px;margin:8px 0;'>"
            "<strong style='color:#111;'>📍 " + loc["name"] + "</strong><br>"
            "<span style='color:#555;font-size:0.85em;'>Lat: " + str(loc["lat"]) + ", Lon: " + str(loc["lon"]) + "</span>"
            "</div>"
        )
    return (
        "<div style='font-family:Arial;padding:10px;'>"
        "<h3 style='color:" + color + ";'>Nearest " + label + "</h3>"
        "<p style='color:#555;'>Galgotias University, Greater Noida area</p>"
        + markers +
        "<div style='margin-top:12px;padding:12px;background:#E3F2FD;border-radius:10px;'>"
        "<strong style='color:#111;'>Tip:</strong> "
        "<span style='color:#333;'>Search " + label + " near me on Google Maps for real-time results!</span>"
        "</div></div>"
    )

def predict_waste(image):
    if image is None:
        return {}, "<p style='color:#333;'>Please upload an image or use webcam.</p>", get_history_html()
    img   = Image.fromarray(image).resize((224, 224))
    arr   = np.expand_dims(np.array(img) / 255.0, axis=0)
    preds = model.predict(arr, verbose=0)
    confs = {classes[i]: float(preds[0][i]) for i in range(len(classes))}
    top   = max(confs, key=confs.get)
    pct   = confs[top] * 100
    status, tip, cat = recycling_tips.get(top, ("Unknown","Unknown","nonrecyclable"))
    co2   = co2_savings.get(top, 0)
    score = impact_scores.get(top, 0)
    fact  = fun_facts.get(top, "")
    classify_count[0] += 1
    total_co2[0]      += co2
    if cat == "recyclable":   bg,border,badge,emoji = "#E8F5E9","#4CAF50","#1B5E20","🟢"
    elif cat == "caution":    bg,border,badge,emoji = "#FFF3E0","#FF9800","#E65100","🟡"
    else:                     bg,border,badge,emoji = "#FFEBEE","#F44336","#B71C1C","🔴"
    score_bar = "🟩" * score + "⬜" * (10 - score)
    history_log.append({"time": datetime.now().strftime("%H:%M:%S"), "item": top.upper(),
                         "confidence": str(round(pct,1)) + "%", "status": status,
                         "emoji": emoji, "co2": co2})
    html = (
        "<div style='background:" + bg + ";border:3px solid " + border + ";border-radius:16px;padding:20px;font-family:Arial;color:#111;'>"
        "<div style='background:" + badge + ";color:white;border-radius:10px;padding:10px 18px;"
        "font-size:1.3em;font-weight:bold;margin-bottom:14px;display:inline-block;'>"
        + emoji + " " + status + " — " + top.upper() + "</div>"
        "<p><strong>Confidence:</strong> " + str(round(pct,1)) + "%</p>"
        "<p><strong>Recycling Tip:</strong> " + tip + "</p>"
        "<p><strong>CO2 Saved:</strong> " + str(co2) + " kg</p>"
        "<hr style='border:1px solid " + border + ";'>"
        "<p><strong>Impact Score:</strong> " + str(score) + "/10 " + score_bar + "</p>"
        "<p style='color:#444;font-size:0.9em;'>💡 " + fact + "</p>"
        "<hr style='border:1px solid " + border + ";'>"
        "<p style='color:#444;font-size:0.85em;'>Total classified: <strong>" + str(classify_count[0]) +
        "</strong> | Total CO2 saved: <strong>" + str(round(total_co2[0],2)) + " kg</strong></p>"
        "</div>"
    )
    return confs, html, get_history_html()

with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classifier") as demo:
    gr.HTML(
        "<div style='text-align:center;padding:22px;background:linear-gradient(135deg,#2E7D32,#1565C0);"
        "border-radius:16px;color:white;margin-bottom:20px;'>"
        "<h1>🌍 Automated Waste Classification System</h1>"
        "<p>Galgotias University | Khushi Dubey &amp; Suhani</p></div>"
    )
    gr.HTML(
        "<div style='display:flex;gap:12px;justify-content:center;margin-bottom:16px;'>"
        "<span style='background:#E8F5E9;border:2px solid #4CAF50;border-radius:8px;padding:6px 14px;color:#111;'>🟢 Recyclable</span>"
        "<span style='background:#FFF3E0;border:2px solid #FF9800;border-radius:8px;padding:6px 14px;color:#111;'>🟡 Check Label</span>"
        "<span style='background:#FFEBEE;border:2px solid #F44336;border-radius:8px;padding:6px 14px;color:#111;'>🔴 Non-Recyclable</span>"
        "</div>"
    )
    with gr.Tabs():
        with gr.Tab("📁 Upload Image"):
            with gr.Row():
                with gr.Column():
                    up_img = gr.Image(label="Upload Waste Image", type="numpy")
                    up_btn = gr.Button("🔍 Classify!", variant="primary", size="lg")
                with gr.Column():
                    up_lbl = gr.Label(num_top_classes=6, label="Confidence Scores")
                    up_res = gr.HTML()
        with gr.Tab("📷 Webcam"):
            with gr.Row():
                with gr.Column():
                    wc_img = gr.Image(label="Point camera at waste item", sources=["webcam"], type="numpy", mirror_webcam=False, interactive=True)
                    wc_btn = gr.Button("🔍 Classify Snapshot!", variant="primary", size="lg")
                with gr.Column():
                    wc_lbl = gr.Label(num_top_classes=6, label="Confidence Scores")
                    wc_res = gr.HTML()
        with gr.Tab("📋 History"):
            hist = gr.HTML(value="<p style='color:#333;text-align:center;padding:20px;'>No classifications yet.</p>")
        with gr.Tab("🗺️ Nearest Bin"):
            bin_radio = gr.Radio(choices=["recyclable","caution","nonrecyclable"], value="recyclable", label="Select waste category")
            map_out   = gr.HTML(value=get_map_html("recyclable"))
            bin_radio.change(fn=get_map_html, inputs=bin_radio, outputs=map_out)
    up_btn.click(predict_waste, inputs=up_img, outputs=[up_lbl, up_res, hist])
    wc_btn.click(predict_waste, inputs=wc_img, outputs=[wc_lbl, wc_res, hist])

demo.launch(share=True)

/tmp/ipykernel_3822/546460998.py:130: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classifier") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://620c193cc7f0b6d12a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classifier") as demo:
    # ... your existing code ...
    with gr.Tab("📷 Webcam"):
        with gr.Row():
            with gr.Column():
                wc_img = gr.Image(
                    label="Point camera at waste item",
                    sources=["webcam"],
                    type="numpy",
                    mirror_webcam=False,
                    interactive=True   # 👈 add this
                )

/tmp/ipykernel_2708/2805493263.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classifier") as demo:


In [ ]:
import numpy as np
from PIL import Image

# Create a fake image to test if model works
fake_image = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
img = Image.fromarray(fake_image).resize((224, 224))
arr = np.expand_dims(np.array(img)/255.0, axis=0)
preds = model.predict(arr, verbose=0)
print("Model output:", preds)
print("Predicted class:", classes[np.argmax(preds)])

NameError: name 'model' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
exec(open("/content/app.py").read())

FileNotFoundError: [Errno 2] No such file or directory: '/content/app.py'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gradio as gr
import numpy as np
from PIL import Image
import tensorflow as tf
import json
from datetime import datetime

model = tf.keras.models.load_model("/content/drive/MyDrive/WasteProject/best_model.keras")
with open("/content/drive/MyDrive/WasteProject/class_names.json", "r") as f:
    class_indices = json.load(f)
classes = [k for k, v in sorted(class_indices.items(), key=lambda x: x[1])]
print("Classes:", classes)

recycling_tips = {
    "cardboard": ("Recyclable",     "Flatten and place in the recycling bin.",   "recyclable"),
    "glass":     ("Recyclable",     "Rinse and place in the glass bin.",         "recyclable"),
    "metal":     ("Recyclable",     "Clean and place in the metal bin.",         "recyclable"),
    "paper":     ("Recyclable",     "Keep dry and place in the paper bin.",      "recyclable"),
    "plastic":   ("Check Label",    "Check recycling number on packaging.",      "caution"),
    "trash":     ("Non-Recyclable", "Place in the general waste bin.",           "nonrecyclable"),
}
co2_savings   = {"cardboard":0.9,"glass":0.3,"metal":1.5,"paper":0.7,"plastic":0.4,"trash":0.0}
impact_scores = {"cardboard":8,"glass":7,"metal":9,"paper":8,"plastic":5,"trash":0}
fun_facts = {
    "cardboard": "Recycling 1 ton of cardboard saves 17 trees!",
    "glass":     "Glass takes ~1 million years to decompose in landfill.",
    "metal":     "Aluminium can be recycled infinitely without quality loss!",
    "paper":     "Recycling 1 ton of paper saves ~26,000 litres of water.",
    "plastic":   "Plastic takes up to 500 years to decompose.",
    "trash":     "Proper waste management can cut greenhouse gases by 20%.",
}
bin_locations = {
    "recyclable":    [{"name":"Blue Recycling Bin - Sector 1","lat":28.4744,"lon":77.5040},
                      {"name":"Blue Recycling Bin - Sector 4","lat":28.4780,"lon":77.5060},
                      {"name":"Community Recycle Center","lat":28.4720,"lon":77.5020}],
    "caution":       [{"name":"Plastic Collection Point","lat":28.4744,"lon":77.5040},
                      {"name":"Plastic Recycling Drop-off","lat":28.4760,"lon":77.5070}],
    "nonrecyclable": [{"name":"General Waste Bin - Block A","lat":28.4730,"lon":77.5030},
                      {"name":"General Waste Bin - Block C","lat":28.4755,"lon":77.5055},
                      {"name":"Municipal Waste Collection","lat":28.4710,"lon":77.5010}],
}

history_log    = []
classify_count = [0]
total_co2      = [0.0]

def get_history_html():
    if not history_log:
        return "<p style='color:#333;text-align:center;padding:20px;'>No classifications yet.</p>"
    rows = ""
    for i, e in enumerate(reversed(history_log[-10:])):
        bg = "#f0f0f0" if i % 2 == 0 else "#ffffff"
        rows += f"""<tr style='background:{bg};'>
            <td style='padding:10px;color:#111;text-align:center;'>{e['time']}</td>
            <td style='padding:10px;color:#111;text-align:center;font-weight:bold;'>{e['item']}</td>
            <td style='padding:10px;color:#111;text-align:center;'>{e['confidence']}</td>
            <td style='padding:10px;color:#111;text-align:center;'>{e['emoji']} {e['status']}</td>
            <td style='padding:10px;color:#111;text-align:center;'>{e['co2']} kg</td>
        </tr>"""
    return f"""
    <div style='font-family:Arial;padding:10px;'>
        <h3 style='color:#2E7D32;'>Classification History (Last 10)</h3>
        <table style='width:100%;border-collapse:collapse;border:1px solid #ddd;'>
            <thead>
                <tr style='background:#2E7D32;color:white;'>
                    <th style='padding:10px;'>Time</th>
                    <th style='padding:10px;'>Item</th>
                    <th style='padding:10px;'>Confidence</th>
                    <th style='padding:10px;'>Status</th>
                    <th style='padding:10px;'>CO2 Saved</th>
                </tr>
            </thead>
            <tbody>{rows}</tbody>
        </table>
        <p style='color:#333;margin-top:10px;'>Total CO2 saved: <strong>{total_co2[0]:.2f} kg</strong></p>
    </div>"""

def get_map_html(category):
    locs  = bin_locations.get(category, bin_locations["nonrecyclable"])
    label = "Recycling Bins" if category=="recyclable" else "Plastic Drop-off Points" if category=="caution" else "General Waste Bins"
    color = "#1B5E20" if category=="recyclable" else "#E65100" if category=="caution" else "#B71C1C"
    markers = "".join([f"""
        <div style='background:white;border:2px solid #4CAF50;border-radius:10px;
                    padding:12px;margin:8px 0;'>
            <strong style='color:#111;'>📍 {l['name']}</strong><br>
            <span style='color:#555;font-size:0.85em;'>Lat: {l['lat']}, Lon: {l['lon']}</span>
        </div>""" for l in locs])
    return f"""
    <div style='font-family:Arial;padding:10px;'>
        <h3 style='color:{color};'>Nearest {label}</h3>
        <p style='color:#555;'>Galgotias University, Greater Noida area</p>
        {markers}
        <div style='margin-top:12px;padding:12px;background:#E3F2FD;border-radius:10px;'>
            <strong style='color:#111;'>Tip:</strong>
            <span style='color:#333;'> Search "{label} near me" on Google Maps!</span>
        </div>
    </div>"""

def predict_waste(image):
    if image is None:
        return {}, "<p style='color:#333;'>Please upload an image or use webcam.</p>", get_history_html()
    img  = Image.fromarray(image).resize((224, 224))
    arr  = np.expand_dims(np.array(img) / 255.0, axis=0)
    preds = model.predict(arr, verbose=0)
    confs = {classes[i]: float(preds[0][i]) for i in range(len(classes))}
    top   = max(confs, key=confs.get)
    conf_pct = confs[top] * 100
    status, tip, cat = recycling_tips.get(top, ("Unknown","Unknown","nonrecyclable"))
    co2   = co2_savings.get(top, 0)
    score = impact_scores.get(top, 0)
    fact  = fun_facts.get(top, "")
    classify_count[0] += 1
    total_co2[0]      += co2
    if cat == "recyclable": bg,border,badge,emoji = "#E8F5E9","#4CAF50","#1B5E20","🟢"
    elif cat == "caution":  bg,border,badge,emoji = "#FFF3E0","#FF9800","#E65100","🟡"
    else:                   bg,border,badge,emoji = "#FFEBEE","#F44336","#B71C1C","🔴"
    score_bar = "🟩" * score + "⬜" * (10 - score)
    history_log.append({
        "time": datetime.now().strftime("%H:%M:%S"),
        "item": top.upper(), "confidence": f"{conf_pct:.1f}%",
        "status": status, "emoji": emoji, "co2": co2
    })
    html = f"""
    <div style='background:{bg};border:3px solid {border};border-radius:16px;
                padding:20px;font-family:Arial;color:#111;'>
        <div style='background:{badge};color:white;border-radius:10px;padding:10px 18px;
                    font-size:1.3em;font-weight:bold;margin-bottom:14px;display:inline-block;'>
            {emoji} {status} — {top.upper()}
        </div>
        <p><strong>Confidence:</strong> {conf_pct:.1f}%</p>
        <p><strong>Tip:</strong> {tip}</p>
        <p><strong>CO2 Saved:</strong> {co2} kg</p>
        <hr style='border:1px solid {border};'>
        <p><strong>Impact Score:</strong> {score}/10 {score_bar}</p>
        <p style='color:#444;font-size:0.9em;'>💡 {fact}</p>
        <hr style='border:1px solid {border};'>
        <p style='color:#444;font-size:0.85em;'>
            Total classified: <strong>{classify_count[0]}</strong> |
            Total CO2 saved: <strong>{total_co2[0]:.2f} kg</strong>
        </p>
    </div>"""
    return confs, html, get_history_html()

with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classifier") as demo:
    gr.HTML("""
    <div style='text-align:center;padding:22px;
                background:linear-gradient(135deg,#2E7D32,#1565C0);
                border-radius:16px;color:white;margin-bottom:20px;'>
        <h1>🌍 Automated Waste Classification System</h1>
        <p>Galgotias University | Khushi Dubey &amp; Suhani</p>
    </div>""")
    gr.HTML("""
    <div style='display:flex;gap:12px;justify-content:center;margin-bottom:16px;'>
        <span style='background:#E8F5E9;border:2px solid #4CAF50;border-radius:8px;padding:6px 14px;color:#111;'>🟢 Recyclable</span>
        <span style='background:#FFF3E0;border:2px solid #FF9800;border-radius:8px;padding:6px 14px;color:#111;'>🟡 Check Label</span>
        <span style='background:#FFEBEE;border:2px solid #F44336;border-radius:8px;padding:6px 14px;color:#111;'>🔴 Non-Recyclable</span>
    </div>""")
    with gr.Tabs():
        with gr.Tab("📁 Upload Image"):
            with gr.Row():
                with gr.Column():
                    up_img = gr.Image(label="Upload Waste Image", type="numpy")
                    up_btn = gr.Button("🔍 Classify!", variant="primary", size="lg")
                with gr.Column():
                    up_lbl = gr.Label(num_top_classes=6, label="Confidence Scores")
                    up_res = gr.HTML()
        with gr.Tab("📷 Webcam"):
            with gr.Row():
                with gr.Column():
                    wc_img = gr.Image(
                        label="Point camera at waste item",
                        sources=["webcam"],
                        type="numpy",
                        mirror_webcam=False,
                        interactive=True
                    )
                    wc_btn = gr.Button("🔍 Classify Snapshot!", variant="primary", size="lg")
                with gr.Column():
                    wc_lbl = gr.Label(num_top_classes=6, label="Confidence Scores")
                    wc_res = gr.HTML()
        with gr.Tab("📋 History"):
            hist = gr.HTML(value="<p style='color:#333;text-align:center;padding:20px;'>No classifications yet.</p>")
        with gr.Tab("🗺️ Nearest Bin"):
            bin_radio = gr.Radio(choices=["recyclable","caution","nonrecyclable"],
                                 value="recyclable", label="Select waste category")
            map_out = gr.HTML(value=get_map_html("recyclable"))
            bin_radio.change(fn=get_map_html, inputs=bin_radio, outputs=map_out)
    up_btn.click(predict_waste, inputs=up_img,  outputs=[up_lbl, up_res, hist])
    wc_btn.click(predict_waste, inputs=wc_img,  outputs=[wc_lbl, wc_res, hist])

demo.launch(share=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


/tmp/ipykernel_3822/2899527738.py:147: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classifier") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3590bac2b3a4edc04d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gradio as gr
import numpy as np
from PIL import Image
import tensorflow as tf
import json
from datetime import datetime

model = tf.keras.models.load_model("/content/drive/MyDrive/WasteProject/best_model.keras")
with open("/content/drive/MyDrive/WasteProject/class_names.json", "r") as f:
    class_indices = json.load(f)
classes = [k for k, v in sorted(class_indices.items(), key=lambda x: x[1])]
print("Classes:", classes)

recycling_tips = {
    "cardboard": ("Recyclable",     "Flatten and place in the recycling bin.",   "recyclable"),
    "glass":     ("Recyclable",     "Rinse and place in the glass bin.",         "recyclable"),
    "metal":     ("Recyclable",     "Clean and place in the metal bin.",         "recyclable"),
    "paper":     ("Recyclable",     "Keep dry and place in the paper bin.",      "recyclable"),
    "plastic":   ("Check Label",    "Check recycling number on packaging.",      "caution"),
    "trash":     ("Non-Recyclable", "Place in the general waste bin.",           "nonrecyclable"),
}
co2_savings   = {"cardboard":0.9,"glass":0.3,"metal":1.5,"paper":0.7,"plastic":0.4,"trash":0.0}
impact_scores = {"cardboard":8,"glass":7,"metal":9,"paper":8,"plastic":5,"trash":0}
fun_facts = {
    "cardboard": "Recycling 1 ton of cardboard saves 17 trees!",
    "glass":     "Glass takes ~1 million years to decompose in landfill.",
    "metal":     "Aluminium can be recycled infinitely without quality loss!",
    "paper":     "Recycling 1 ton of paper saves ~26,000 litres of water.",
    "plastic":   "Plastic takes up to 500 years to decompose.",
    "trash":     "Proper waste management can cut greenhouse gases by 20%.",
}
bin_locations = {
    "recyclable":    [{"name":"Blue Recycling Bin - Sector 1","lat":28.4744,"lon":77.5040},
                      {"name":"Blue Recycling Bin - Sector 4","lat":28.4780,"lon":77.5060},
                      {"name":"Community Recycle Center","lat":28.4720,"lon":77.5020}],
    "caution":       [{"name":"Plastic Collection Point","lat":28.4744,"lon":77.5040},
                      {"name":"Plastic Recycling Drop-off","lat":28.4760,"lon":77.5070}],
    "nonrecyclable": [{"name":"General Waste Bin - Block A","lat":28.4730,"lon":77.5030},
                      {"name":"General Waste Bin - Block C","lat":28.4755,"lon":77.5055},
                      {"name":"Municipal Waste Collection","lat":28.4710,"lon":77.5010}],
}

history_log    = []
classify_count = [0]
total_co2      = [0.0]

def get_history_html():
    if not history_log:
        return "<p style='color:#333;text-align:center;padding:20px;'>No classifications yet.</p>"
    rows = ""
    for i, e in enumerate(reversed(history_log[-10:])):
        bg = "#f0f0f0" if i % 2 == 0 else "#ffffff"
        rows += f"""<tr style='background:{bg};'>
            <td style='padding:10px;color:#111;text-align:center;'>{e['time']}</td>
            <td style='padding:10px;color:#111;text-align:center;font-weight:bold;'>{e['item']}</td>
            <td style='padding:10px;color:#111;text-align:center;'>{e['confidence']}</td>
            <td style='padding:10px;color:#111;text-align:center;'>{e['emoji']} {e['status']}</td>
            <td style='padding:10px;color:#111;text-align:center;'>{e['co2']} kg</td>
        </tr>"""
    return f"""
    <div style='font-family:Arial;padding:10px;'>
        <h3 style='color:#2E7D32;'>Classification History (Last 10)</h3>
        <table style='width:100%;border-collapse:collapse;border:1px solid #ddd;'>
            <thead>
                <tr style='background:#2E7D32;color:white;'>
                    <th style='padding:10px;'>Time</th>
                    <th style='padding:10px;'>Item</th>
                    <th style='padding:10px;'>Confidence</th>
                    <th style='padding:10px;'>Status</th>
                    <th style='padding:10px;'>CO2 Saved</th>
                </tr>
            </thead>
            <tbody>{rows}</tbody>
        </table>
        <p style='color:#333;margin-top:10px;'>Total CO2 saved: <strong>{total_co2[0]:.2f} kg</strong></p>
    </div>"""

def get_map_html(category):
    locs  = bin_locations.get(category, bin_locations["nonrecyclable"])
    label = "Recycling Bins" if category=="recyclable" else "Plastic Drop-off Points" if category=="caution" else "General Waste Bins"
    color = "#1B5E20" if category=="recyclable" else "#E65100" if category=="caution" else "#B71C1C"
    markers = "".join([f"""
        <div style='background:white;border:2px solid #4CAF50;border-radius:10px;
                    padding:12px;margin:8px 0;'>
            <strong style='color:#111;'>📍 {l['name']}</strong><br>
            <span style='color:#555;font-size:0.85em;'>Lat: {l['lat']}, Lon: {l['lon']}</span>
        </div>""" for l in locs])
    return f"""
    <div style='font-family:Arial;padding:10px;'>
        <h3 style='color:{color};'>Nearest {label}</h3>
        <p style='color:#555;'>Galgotias University, Greater Noida area</p>
        {markers}
        <div style='margin-top:12px;padding:12px;background:#E3F2FD;border-radius:10px;'>
            <strong style='color:#111;'>Tip:</strong>
            <span style='color:#333;'> Search "{label} near me" on Google Maps!</span>
        </div>
    </div>"""

def predict_waste(image):
    if image is None:
        return {}, "<p style='color:#333;'>Please upload an image or use webcam.</p>", get_history_html()
    img  = Image.fromarray(image).resize((224, 224))
    arr  = np.expand_dims(np.array(img) / 255.0, axis=0)
    preds = model.predict(arr, verbose=0)
    confs = {classes[i]: float(preds[0][i]) for i in range(len(classes))}
    top   = max(confs, key=confs.get)
    conf_pct = confs[top] * 100
    status, tip, cat = recycling_tips.get(top, ("Unknown","Unknown","nonrecyclable"))
    co2   = co2_savings.get(top, 0)
    score = impact_scores.get(top, 0)
    fact  = fun_facts.get(top, "")
    classify_count[0] += 1
    total_co2[0]      += co2
    if cat == "recyclable": bg,border,badge,emoji = "#E8F5E9","#4CAF50","#1B5E20","🟢"
    elif cat == "caution":  bg,border,badge,emoji = "#FFF3E0","#FF9800","#E65100","🟡"
    else:                   bg,border,badge,emoji = "#FFEBEE","#F44336","#B71C1C","🔴"
    score_bar = "🟩" * score + "⬜" * (10 - score)
    history_log.append({
        "time": datetime.now().strftime("%H:%M:%S"),
        "item": top.upper(), "confidence": f"{conf_pct:.1f}%",
        "status": status, "emoji": emoji, "co2": co2
    })
    html = f"""
    <div style='background:{bg};border:3px solid {border};border-radius:16px;
                padding:20px;font-family:Arial;color:#111;'>
        <div style='background:{badge};color:white;border-radius:10px;padding:10px 18px;
                    font-size:1.3em;font-weight:bold;margin-bottom:14px;display:inline-block;'>
            {emoji} {status} — {top.upper()}
        </div>
        <p><strong>Confidence:</strong> {conf_pct:.1f}%</p>
        <p><strong>Tip:</strong> {tip}</p>
        <p><strong>CO2 Saved:</strong> {co2} kg</p>
        <hr style='border:1px solid {border};'>
        <p><strong>Impact Score:</strong> {score}/10 {score_bar}</p>
        <p style='color:#444;font-size:0.9em;'>💡 {fact}</p>
        <hr style='border:1px solid {border};'>
        <p style='color:#444;font-size:0.85em;'>
            Total classified: <strong>{classify_count[0]}</strong> |
            Total CO2 saved: <strong>{total_co2[0]:.2f} kg</strong>
        </p>
    </div>"""
    return confs, html, get_history_html()

with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classifier") as demo:
    gr.HTML("""
    <div style='text-align:center;padding:22px;
                background:linear-gradient(135deg,#2E7D32,#1565C0);
                border-radius:16px;color:white;margin-bottom:20px;'>
        <h1>🌍 Automated Waste Classification System</h1>
        <p>Galgotias University | Khushi Dubey &amp; Suhani</p>
    </div>""")
    gr.HTML("""
    <div style='display:flex;gap:12px;justify-content:center;margin-bottom:16px;'>
        <span style='background:#E8F5E9;border:2px solid #4CAF50;border-radius:8px;padding:6px 14px;color:#111;'>🟢 Recyclable</span>
        <span style='background:#FFF3E0;border:2px solid #FF9800;border-radius:8px;padding:6px 14px;color:#111;'>🟡 Check Label</span>
        <span style='background:#FFEBEE;border:2px solid #F44336;border-radius:8px;padding:6px 14px;color:#111;'>🔴 Non-Recyclable</span>
    </div>""")
    with gr.Tabs():
        with gr.Tab("📁 Upload Image"):
            with gr.Row():
                with gr.Column():
                    up_img = gr.Image(label="Upload Waste Image", type="numpy")
                    up_btn = gr.Button("🔍 Classify!", variant="primary", size="lg")
                with gr.Column():
                    up_lbl = gr.Label(num_top_classes=6, label="Confidence Scores")
                    up_res = gr.HTML()
        with gr.Tab("📷 Webcam"):
            with gr.Row():
                with gr.Column():
                    wc_img = gr.Image(
                        label="Point camera at waste item",
                        sources=["webcam"],
                        type="numpy",
                        mirror_webcam=False,
                        interactive=True
                    )
                    wc_btn = gr.Button("🔍 Classify Snapshot!", variant="primary", size="lg")
                with gr.Column():
                    wc_lbl = gr.Label(num_top_classes=6, label="Confidence Scores")
                    wc_res = gr.HTML()
        with gr.Tab("📋 History"):
            hist = gr.HTML(value="<p style='color:#333;text-align:center;padding:20px;'>No classifications yet.</p>")
        with gr.Tab("🗺️ Nearest Bin"):
            bin_radio = gr.Radio(choices=["recyclable","caution","nonrecyclable"],
                                 value="recyclable", label="Select waste category")
            map_out = gr.HTML(value=get_map_html("recyclable"))
            bin_radio.change(fn=get_map_html, inputs=bin_radio, outputs=map_out)
    up_btn.click(predict_waste, inputs=up_img,  outputs=[up_lbl, up_res, hist])
    wc_btn.click(predict_waste, inputs=wc_img,  outputs=[wc_lbl, wc_res, hist])

demo.launch(share=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


/tmp/ipykernel_3822/2899527738.py:147: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classifier") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eaba4a71003a564a23.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
import json

# Save app.py to Drive
app_code = '''
import gradio as gr
import numpy as np
from PIL import Image
import tensorflow as tf
import json
from datetime import datetime

model = tf.keras.models.load_model("best_model.keras")
with open("class_names.json", "r") as f:
    class_indices = json.load(f)
classes = [k for k, v in sorted(class_indices.items(), key=lambda x: x[1])]

recycling_tips = {
    "cardboard": ("Recyclable",     "Flatten and place in the recycling bin.",   "recyclable"),
    "glass":     ("Recyclable",     "Rinse and place in the glass bin.",         "recyclable"),
    "metal":     ("Recyclable",     "Clean and place in the metal bin.",         "recyclable"),
    "paper":     ("Recyclable",     "Keep dry and place in the paper bin.",      "recyclable"),
    "plastic":   ("Check Label",    "Check recycling number on packaging.",      "caution"),
    "trash":     ("Non-Recyclable", "Place in the general waste bin.",           "nonrecyclable"),
}
co2_savings   = {"cardboard":0.9,"glass":0.3,"metal":1.5,"paper":0.7,"plastic":0.4,"trash":0.0}
impact_scores = {"cardboard":8,"glass":7,"metal":9,"paper":8,"plastic":5,"trash":0}
fun_facts = {
    "cardboard": "Recycling 1 ton of cardboard saves 17 trees!",
    "glass":     "Glass takes ~1 million years to decompose in landfill.",
    "metal":     "Aluminium can be recycled infinitely without quality loss!",
    "paper":     "Recycling 1 ton of paper saves ~26,000 litres of water.",
    "plastic":   "Plastic takes up to 500 years to decompose.",
    "trash":     "Proper waste management can cut greenhouse gases by 20%.",
}
bin_locations = {
    "recyclable":    [{"name":"Blue Recycling Bin - Sector 1","lat":28.4744,"lon":77.5040},
                      {"name":"Blue Recycling Bin - Sector 4","lat":28.4780,"lon":77.5060},
                      {"name":"Community Recycle Center","lat":28.4720,"lon":77.5020}],
    "caution":       [{"name":"Plastic Collection Point","lat":28.4744,"lon":77.5040},
                      {"name":"Plastic Recycling Drop-off","lat":28.4760,"lon":77.5070}],
    "nonrecyclable": [{"name":"General Waste Bin - Block A","lat":28.4730,"lon":77.5030},
                      {"name":"General Waste Bin - Block C","lat":28.4755,"lon":77.5055},
                      {"name":"Municipal Waste Collection","lat":28.4710,"lon":77.5010}],
}

history_log = []
classify_count = [0]
total_co2 = [0.0]

def get_history_html():
    if not history_log:
        return "<p style=\'color:#333;text-align:center;padding:20px;\'>No classifications yet.</p>"
    rows = ""
    for i, e in enumerate(reversed(history_log[-10:])):
        bg = "#f0f0f0" if i % 2 == 0 else "#ffffff"
        rows += f"<tr style=\'background:{bg};\'><td style=\'padding:10px;color:#111;text-align:center;\'>{e[\'time\']}</td><td style=\'padding:10px;color:#111;text-align:center;font-weight:bold;\'>{e[\'item\']}</td><td style=\'padding:10px;color:#111;text-align:center;\'>{e[\'confidence\']}</td><td style=\'padding:10px;color:#111;text-align:center;\'>{e[\'emoji\']} {e[\'status\']}</td><td style=\'padding:10px;color:#111;text-align:center;\'>{e[\'co2\']} kg</td></tr>"
    return f"<div style=\'font-family:Arial;padding:10px;\'><h3 style=\'color:#2E7D32;\'>Classification History (Last 10)</h3><table style=\'width:100%;border-collapse:collapse;border:1px solid #ddd;\'><thead><tr style=\'background:#2E7D32;color:white;\'><th style=\'padding:10px;\'>Time</th><th style=\'padding:10px;\'>Item</th><th style=\'padding:10px;\'>Confidence</th><th style=\'padding:10px;\'>Status</th><th style=\'padding:10px;\'>CO2 Saved</th></tr></thead><tbody>{rows}</tbody></table><p style=\'color:#333;margin-top:10px;\'>Total CO2 saved: <strong>{total_co2[0]:.2f} kg</strong></p></div>"

def get_map_html(category):
    locs = bin_locations.get(category, bin_locations["nonrecyclable"])
    label = "Recycling Bins" if category=="recyclable" else "Plastic Drop-off Points" if category=="caution" else "General Waste Bins"
    color = "#1B5E20" if category=="recyclable" else "#E65100" if category=="caution" else "#B71C1C"
    markers = "".join([f"<div style=\'background:white;border:2px solid #4CAF50;border-radius:10px;padding:12px;margin:8px 0;\'><strong style=\'color:#111;\'>📍 {l[\'name\']}</strong><br><span style=\'color:#555;font-size:0.85em;\'>Lat: {l[\'lat\']}, Lon: {l[\'lon\']}</span></div>" for l in locs])
    return f"<div style=\'font-family:Arial;padding:10px;\'><h3 style=\'color:{color};\'>Nearest {label}</h3><p style=\'color:#555;\'>Galgotias University, Greater Noida area</p>{markers}<div style=\'margin-top:12px;padding:12px;background:#E3F2FD;border-radius:10px;\'><strong style=\'color:#111;\'>Tip:</strong> <span style=\'color:#333;\'>Search \"{label} near me\" on Google Maps!</span></div></div>"

def predict_waste(image):
    if image is None:
        return {}, "<p style=\'color:#333;\'>Please upload an image or use webcam.</p>", get_history_html()
    img = Image.fromarray(image).resize((224, 224))
    arr = np.expand_dims(np.array(img) / 255.0, axis=0)
    preds = model.predict(arr, verbose=0)
    confs = {classes[i]: float(preds[0][i]) for i in range(len(classes))}
    top = max(confs, key=confs.get)
    conf_pct = confs[top] * 100
    status, tip, cat = recycling_tips.get(top, ("Unknown","Unknown","nonrecyclable"))
    co2 = co2_savings.get(top, 0)
    score = impact_scores.get(top, 0)
    fact = fun_facts.get(top, "")
    classify_count[0] += 1
    total_co2[0] += co2
    if cat == "recyclable": bg,border,badge,emoji = "#E8F5E9","#4CAF50","#1B5E20","🟢"
    elif cat == "caution":  bg,border,badge,emoji = "#FFF3E0","#FF9800","#E65100","🟡"
    else:                   bg,border,badge,emoji = "#FFEBEE","#F44336","#B71C1C","🔴"
    score_bar = "🟩" * score + "⬜" * (10 - score)
    history_log.append({"time":datetime.now().strftime("%H:%M:%S"),"item":top.upper(),"confidence":f"{conf_pct:.1f}%","status":status,"emoji":emoji,"co2":co2})
    html = f"<div style=\'background:{bg};border:3px solid {border};border-radius:16px;padding:20px;font-family:Arial;color:#111;\'><div style=\'background:{badge};color:white;border-radius:10px;padding:10px 18px;font-size:1.3em;font-weight:bold;margin-bottom:14px;display:inline-block;\'>{emoji} {status} — {top.upper()}</div><p><strong>Confidence:</strong> {conf_pct:.1f}%</p><p><strong>Tip:</strong> {tip}</p><p><strong>CO2 Saved:</strong> {co2} kg</p><hr style=\'border:1px solid {border};\'><p><strong>Impact Score:</strong> {score}/10 {score_bar}</p><p style=\'color:#444;font-size:0.9em;\'>💡 {fact}</p><hr style=\'border:1px solid {border};\'><p style=\'color:#444;font-size:0.85em;\'>Total classified: <strong>{classify_count[0]}</strong> | Total CO2 saved: <strong>{total_co2[0]:.2f} kg</strong></p></div>"
    return confs, html, get_history_html()

with gr.Blocks(theme=gr.themes.Soft(), title="Waste Classifier") as demo:
    gr.HTML("<div style=\'text-align:center;padding:22px;background:linear-gradient(135deg,#2E7D32,#1565C0);border-radius:16px;color:white;margin-bottom:20px;\'><h1>🌍 Automated Waste Classification System</h1><p>Galgotias University | Khushi Dubey &amp; Suhani</p></div>")
    gr.HTML("<div style=\'display:flex;gap:12px;justify-content:center;margin-bottom:16px;\'><span style=\'background:#E8F5E9;border:2px solid #4CAF50;border-radius:8px;padding:6px 14px;color:#111;\'>🟢 Recyclable</span><span style=\'background:#FFF3E0;border:2px solid #FF9800;border-radius:8px;padding:6px 14px;color:#111;\'>🟡 Check Label</span><span style=\'background:#FFEBEE;border:2px solid #F44336;border-radius:8px;padding:6px 14px;color:#111;\'>🔴 Non-Recyclable</span></div>")
    with gr.Tabs():
        with gr.Tab("📁 Upload Image"):
            with gr.Row():
                with gr.Column():
                    up_img = gr.Image(label="Upload Waste Image", type="numpy")
                    up_btn = gr.Button("🔍 Classify!", variant="primary", size="lg")
                with gr.Column():
                    up_lbl = gr.Label(num_top_classes=6, label="Confidence Scores")
                    up_res = gr.HTML()
        with gr.Tab("📷 Webcam"):
            with gr.Row():
                with gr.Column():
                    wc_img = gr.Image(label="Point camera at waste item", sources=["webcam"], type="numpy", mirror_webcam=False, interactive=True)
                    wc_btn = gr.Button("🔍 Classify Snapshot!", variant="primary", size="lg")
                with gr.Column():
                    wc_lbl = gr.Label(num_top_classes=6, label="Confidence Scores")
                    wc_res = gr.HTML()
        with gr.Tab("📋 History"):
            hist = gr.HTML(value="<p style=\'color:#333;text-align:center;padding:20px;\'>No classifications yet.</p>")
        with gr.Tab("🗺️ Nearest Bin"):
            bin_radio = gr.Radio(choices=["recyclable","caution","nonrecyclable"], value="recyclable", label="Select waste category")
            map_out = gr.HTML(value=get_map_html("recyclable"))
            bin_radio.change(fn=get_map_html, inputs=bin_radio, outputs=map_out)
    up_btn.click(predict_waste, inputs=up_img, outputs=[up_lbl, up_res, hist])
    wc_btn.click(predict_waste, inputs=wc_img, outputs=[wc_lbl, wc_res, hist])

demo.launch()
'''

# Save app.py to Drive
with open("/content/drive/MyDrive/WasteProject/app.py", "w") as f:
    f.write(app_code)

# Save requirements.txt
requirements = """gradio
tensorflow
numpy
Pillow
"""
with open("/content/drive/MyDrive/WasteProject/requirements.txt", "w") as f:
    f.write(requirements)

# Save README
readme = """# Waste Classification System
By Khushi Dubey & Suhani | Galgotias University

## How to Run on Hugging Face Spaces:
1. Upload all files to a new Hugging Face Space
2. Select Gradio as SDK
3. Upload best_model.keras and class_names.json
4. App will launch automatically!

## Files:
- app.py: Main application
- best_model.keras: Trained model
- class_names.json: Class labels
- requirements.txt: Required libraries
"""
with open("/content/drive/MyDrive/WasteProject/README.md", "w") as f:
    f.write(readme)

print("All files saved to Drive!")
print(os.listdir("/content/drive/MyDrive/WasteProject"))

All files saved to Drive!
['best_model.keras', 'class_names.json', 'app.py', 'requirements.txt', 'README.md']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# Save app.py to Drive
app_code = open("/content/app.py").read() if os.path.exists("/content/app.py") else ""

if not app_code:
    print("app.py nahi mila — Drive se copy karo")
else:
    print("app.py found!")

# Check what's in WasteProject
print(os.listdir("/content/drive/MyDrive/WasteProject"))

Mounted at /content/drive
app.py nahi mila — Drive se copy karo
['best_model.keras', 'class_names.json', 'app.py', 'requirements.txt', 'README.md']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Check if dataset already exists
if os.path.exists("garbage_data"):
    print("Dataset already present!")
else:
    os.system("kaggle datasets download -d asdasdasasdas/garbage-classification")
    os.system("unzip -q garbage-classification.zip -d garbage_data")
    print("Dataset downloaded!")

print(os.listdir("garbage_data/Garbage classification/Garbage classification/"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset already present!
['plastic', 'metal', 'glass', 'trash', 'cardboard', 'paper']


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

os.makedirs('/content/drive/MyDrive/WasteProject', exist_ok=True)

DATASET_PATH = "garbage_data/Garbage classification/Garbage classification/"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(DATASET_PATH,
    target_size=(224,224), batch_size=32, class_mode='categorical',
    subset='training', seed=42)

val_gen = train_datagen.flow_from_directory(DATASET_PATH,
    target_size=(224,224), batch_size=32, class_mode='categorical',
    subset='validation', seed=42)

# ── EfficientNetB0 + GRU ──────────────────────────────────
base = applications.EfficientNetB0(
    weights='imagenet', include_top=False, input_shape=(224,224,3)
)
base.trainable = False  # Freeze base first

inputs = layers.Input(shape=(224,224,3))
x = base(inputs, training=False)

# Reshape for GRU — flatten spatial dims into sequence
x = layers.Reshape((-1, x.shape[-1]))(x)  # (batch, steps, features)

# GRU instead of BiLSTM
x = layers.Bidirectional(layers.GRU(256, return_sequences=True))(x)
x = layers.Bidirectional(layers.GRU(128))(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(6, activation='softmax')(x)

model_efficient_gru = models.Model(inputs, outputs)
model_efficient_gru.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_efficient_gru.summary()

Found 2024 images belonging to 6 classes.
Found 503 images belonging to 6 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 49, 1280)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 49, 512)        │     2,362,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       493,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,973,353 (26.60 MB)

 Trainable params: 2,923,270 (11.15 MB)

 Non-trainable params: 4,050,083 (15.45 MB)

In [ ]:
import json

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    ModelCheckpoint(
        '/content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

print("Training EfficientNetB0 + GRU...")
history = model_efficient_gru.fit(
    train_gen, epochs=10,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

# Save class names
with open('/content/drive/MyDrive/WasteProject/class_names.json', 'w') as f:
    json.dump(train_gen.class_indices, f)

print("\nFinal Results:")
print(f"Best Val Accuracy: {max(history.history['val_accuracy'])*100:.2f}%")
print("Files saved:", os.listdir('/content/drive/MyDrive/WasteProject'))

Training EfficientNetB0 + GRU...
Epoch 1/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.1667 - loss: 2.0037
Epoch 1: val_accuracy improved from None to 0.15905, saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras

Epoch 1: finished saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 318s 5s/step - accuracy: 0.1902 - loss: 1.8750 - val_accuracy: 0.1590 - val_loss: 2.2523
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.2299 - loss: 1.7390
Epoch 2: val_accuracy improved from 0.15905 to 0.16302, saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras

Epoch 2: finished saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 311s 5s/step - accuracy: 0.2268 - loss: 1.7353 - val_accuracy: 0.1630 - val_loss: 2.1554
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.2236 - loss: 1.

In [ ]:
# EfficientNetB0 + GRU — Fixed Version
base = applications.EfficientNetB0(
    weights='imagenet', include_top=False, input_shape=(224,224,3)
)
base.trainable = False

inputs = layers.Input(shape=(224,224,3))
x = base(inputs, training=False)

# GlobalAveragePooling first — then repeat for GRU
x = layers.GlobalAveragePooling2D()(x)          # (batch, 1280)
x = layers.RepeatVector(7)(x)                    # (batch, 7, 1280)

# Bidirectional GRU
x = layers.Bidirectional(layers.GRU(256, return_sequences=False))(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(6, activation='softmax')(x)

model_efficient_gru = models.Model(inputs, outputs)
model_efficient_gru.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_efficient_gru.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 7, 1280)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 512)            │     2,362,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,545,833 (24.97 MB)

 Trainable params: 2,495,750 (9.52 MB)

 Non-trainable params: 4,050,083 (15.45 MB)

In [ ]:
import json

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    ModelCheckpoint(
        '/content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

print("Training EfficientNetB0 + GRU...")
history = model_efficient_gru.fit(
    train_gen, epochs=10,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

# Save class names
with open('/content/drive/MyDrive/WasteProject/class_names.json', 'w') as f:
    json.dump(train_gen.class_indices, f)

print("\nFinal Results:")
print(f"Best Val Accuracy: {max(history.history['val_accuracy'])*100:.2f}%")
print("Files saved:", os.listdir('/content/drive/MyDrive/WasteProject'))

Training EfficientNetB0 + GRU...
Epoch 1/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.1843 - loss: 2.0228
Epoch 1: val_accuracy improved from None to 0.05368, saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras

Epoch 1: finished saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 255s 4s/step - accuracy: 0.1823 - loss: 1.9909 - val_accuracy: 0.0537 - val_loss: 2.5322
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.2085 - loss: 1.8702
Epoch 2: val_accuracy improved from 0.05368 to 0.23459, saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras

Epoch 2: finished saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 256s 3s/step - accuracy: 0.2075 - loss: 1.8673 - val_accuracy: 0.2346 - val_loss: 2.4912
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.1862 - loss: 1.

In [ ]:
# EfficientNetB0 + GRU — Properly Fixed
base = applications.EfficientNetB0(
    weights='imagenet', include_top=False, input_shape=(224,224,3)
)

# Unfreeze last 20 layers — important!
for layer in base.layers[:-20]:
    layer.trainable = False
for layer in base.layers[-20:]:
    layer.trainable = True

inputs = layers.Input(shape=(224,224,3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)    # (batch, 1280)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(6, activation='softmax')(x)

model_efficient_gru = models.Model(inputs, outputs)
model_efficient_gru.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),  # lower learning rate
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_efficient_gru.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,049 (16.70 MB)

 Trainable params: 1,680,438 (6.41 MB)

 Non-trainable params: 2,698,611 (10.29 MB)

In [ ]:
import json

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    ModelCheckpoint(
        '/content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

print("Training EfficientNetB0 + GRU...")
history = model_efficient_gru.fit(
    train_gen, epochs=10,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

# Save class names
with open('/content/drive/MyDrive/WasteProject/class_names.json', 'w') as f:
    json.dump(train_gen.class_indices, f)

print("\nFinal Results:")
print(f"Best Val Accuracy: {max(history.history['val_accuracy'])*100:.2f}%")
print("Files saved:", os.listdir('/content/drive/MyDrive/WasteProject'))

Training EfficientNetB0 + GRU...
Epoch 1/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.1939 - loss: 1.7728
Epoch 1: val_accuracy improved from None to 0.23459, saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras

Epoch 1: finished saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 256s 4s/step - accuracy: 0.1971 - loss: 1.7609 - val_accuracy: 0.2346 - val_loss: 1.7581
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.2108 - loss: 1.7561
Epoch 2: val_accuracy did not improve from 0.23459
64/64 ━━━━━━━━━━━━━━━━━━━━ 227s 4s/step - accuracy: 0.2134 - loss: 1.7485 - val_accuracy: 0.2346 - val_loss: 1.7452
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.2431 - loss: 1.7354
Epoch 3: val_accuracy did not improve from 0.23459
64/64 ━━━━━━━━━━━━━━━━━━━━ 234s 4s/step - accuracy: 0.2213 - loss: 1.7450 - val_accuracy: 0.2346 - val_loss: 1.7246
Epoch 4/10
64/64 ━━━━━━━

In [ ]:
# Correct EfficientNetB0 preprocessing + proper unfreeze
from tensorflow.keras.applications.efficientnet import preprocess_input

DATASET_PATH = "garbage_data/Garbage classification/Garbage classification/"

# EfficientNet ka apna preprocessing chahiye
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.2,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(DATASET_PATH,
    target_size=(224,224), batch_size=32, class_mode='categorical',
    subset='training', seed=42)

val_gen = train_datagen.flow_from_directory(DATASET_PATH,
    target_size=(224,224), batch_size=32, class_mode='categorical',
    subset='validation', seed=42)

# Model
base = applications.EfficientNetB0(
    weights='imagenet', include_top=False, input_shape=(224,224,3)
)
base.trainable = False

inputs = layers.Input(shape=(224,224,3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(6, activation='softmax')(x)

model_efficient_gru = models.Model(inputs, outputs)
model_efficient_gru.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("Model ready!")
model_efficient_gru.summary()

Found 2024 images belonging to 6 classes.
Found 503 images belonging to 6 classes.
Model ready!


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,384,169 (16.72 MB)

 Trainable params: 332,038 (1.27 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [ ]:
import json

# Phase 1 — Train only top layers
print("Phase 1 — Training top layers...")
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    ModelCheckpoint(
        '/content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

history1 = model_efficient_gru.fit(
    train_gen, epochs=5,
    validation_data=val_gen,
    callbacks=callbacks, verbose=1
)

# Phase 2 — Unfreeze and fine-tune
print("\nPhase 2 — Fine-tuning...")
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model_efficient_gru.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),  # much lower lr
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model_efficient_gru.fit(
    train_gen, epochs=10,
    validation_data=val_gen,
    callbacks=callbacks, verbose=1
)

# Save
with open('/content/drive/MyDrive/WasteProject/class_names.json', 'w') as f:
    json.dump(train_gen.class_indices, f)

all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
print(f"\nBest Val Accuracy: {max(all_val_acc)*100:.2f}%")
print("Files:", os.listdir('/content/drive/MyDrive/WasteProject'))

Phase 1 — Training top layers...
Epoch 1/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6201 - loss: 1.1362
Epoch 1: val_accuracy improved from None to 0.81511, saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras

Epoch 1: finished saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 236s 3s/step - accuracy: 0.7352 - loss: 0.8137 - val_accuracy: 0.8151 - val_loss: 0.6402
Epoch 2/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8678 - loss: 0.3703
Epoch 2: val_accuracy improved from 0.81511 to 0.81710, saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras

Epoch 2: finished saving model to /content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 214s 3s/step - accuracy: 0.8790 - loss: 0.3522 - val_accuracy: 0.8171 - val_loss: 0.5425
Epoch 3/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9134 - loss: 0.261

In [ ]:
# Load original model
original_model = tf.keras.models.load_model(
    '/content/drive/MyDrive/WasteProject/best_model.keras'
)

# Evaluate both
print("Evaluating Original Model (MobileNetV2)...")
orig_loss, orig_acc = original_model.evaluate(val_gen, verbose=0)

print("Evaluating New Model (EfficientNetB0)...")
new_loss, new_acc = model_efficient_gru.evaluate(val_gen, verbose=0)

print("\n===== COMPARISON =====")
print(f"Original MobileNetV2:     {orig_acc*100:.2f}%")
print(f"New EfficientNetB0:       {new_acc*100:.2f}%")
print(f"Improvement:              {(new_acc - orig_acc)*100:+.2f}%")

NameError: name 'tf' is not defined

In [ ]:
import tensorflow as tf

In [ ]:
# Load original model
original_model = tf.keras.models.load_model(
    '/content/drive/MyDrive/WasteProject/best_model.keras'
)

# Evaluate both
print("Evaluating Original Model (MobileNetV2)...")
orig_loss, orig_acc = original_model.evaluate(val_gen, verbose=0)

print("Evaluating New Model (EfficientNetB0)...")
new_loss, new_acc = model_efficient_gru.evaluate(val_gen, verbose=0)

print("\n===== COMPARISON =====")
print(f"Original MobileNetV2:     {orig_acc*100:.2f}%")
print(f"New EfficientNetB0:       {new_acc*100:.2f}%")
print(f"Improvement:              {(new_acc - orig_acc)*100:+.2f}%")

ValueError: File not found: filepath=/content/drive/MyDrive/WasteProject/best_model.keras. Please ensure the file is an accessible `.keras` zip file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import tensorflow as tf

# Check files
import os
print(os.listdir('/content/drive/MyDrive/WasteProject'))

Mounted at /content/drive
['best_model.keras', 'app.py', 'requirements.txt', 'README.md', 'best_model_efficientnet_gru.keras', 'class_names.json']


In [ ]:
# Load original model
original_model = tf.keras.models.load_model(
    '/content/drive/MyDrive/WasteProject/best_model.keras'
)

# Evaluate both
print("Evaluating Original Model (MobileNetV2)...")
orig_loss, orig_acc = original_model.evaluate(val_gen, verbose=0)

print("Evaluating New Model (EfficientNetB0)...")
new_loss, new_acc = model_efficient_gru.evaluate(val_gen, verbose=0)

print("\n===== COMPARISON =====")
print(f"Original MobileNetV2:     {orig_acc*100:.2f}%")
print(f"New EfficientNetB0:       {new_acc*100:.2f}%")
print(f"Improvement:              {(new_acc - orig_acc)*100:+.2f}%")

Evaluating Original Model (MobileNetV2)...


NameError: name 'val_gen' is not defined

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

DATASET_PATH = "garbage_data/Garbage classification/Garbage classification/"

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

val_gen = val_datagen.flow_from_directory(DATASET_PATH,
    target_size=(224,224), batch_size=32, class_mode='categorical',
    subset='validation', seed=42)

# Load both models
original_model = tf.keras.models.load_model('/content/drive/MyDrive/WasteProject/best_model.keras')
new_model = tf.keras.models.load_model('/content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras')

# Compare
orig_loss, orig_acc = original_model.evaluate(val_gen, verbose=0)
new_loss, new_acc = new_model.evaluate(val_gen, verbose=0)

print("\n===== COMPARISON =====")
print(f"Original MobileNetV2:   {orig_acc*100:.2f}%")
print(f"New EfficientNetB0:     {new_acc*100:.2f}%")
print(f"Improvement:            {(new_acc - orig_acc)*100:+.2f}%")

FileNotFoundError: [Errno 2] No such file or directory: 'garbage_data/Garbage classification/Garbage classification/'

In [ ]:
import shutil
shutil.copytree(
    "garbage_data",
    "/content/drive/MyDrive/WasteProject/garbage_data"
)
print("Dataset saved to Drive!")

FileNotFoundError: [Errno 2] No such file or directory: 'garbage_data'

In [ ]:
import os
os.system("kaggle datasets download -d asdasdasasdas/garbage-classification")
os.system("unzip -q garbage-classification.zip -d garbage_data")
print(os.listdir("garbage_data/Garbage classification/Garbage classification/"))

['plastic', 'metal', 'glass', 'trash', 'cardboard', 'paper']


In [ ]:
# Load original model
original_model = tf.keras.models.load_model(
    '/content/drive/MyDrive/WasteProject/best_model.keras'
)

# Evaluate both
print("Evaluating Original Model (MobileNetV2)...")
orig_loss, orig_acc = original_model.evaluate(val_gen, verbose=0)

print("Evaluating New Model (EfficientNetB0)...")
new_loss, new_acc = model_efficient_gru.evaluate(val_gen, verbose=0)

print("\n===== COMPARISON =====")
print(f"Original MobileNetV2:     {orig_acc*100:.2f}%")
print(f"New EfficientNetB0:       {new_acc*100:.2f}%")
print(f"Improvement:              {(new_acc - orig_acc)*100:+.2f}%")

Evaluating Original Model (MobileNetV2)...


NameError: name 'val_gen' is not defined

In [ ]:
import os
os.system("kaggle datasets download -d asdasdasasdas/garbage-classification")
os.system("unzip -q garbage-classification.zip -d garbage_data")
print("Done!")

Done!


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

DATASET_PATH = "garbage_data/Garbage classification/Garbage classification/"

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
).flow_from_directory(DATASET_PATH,
    target_size=(224,224), batch_size=32, class_mode='categorical',
    subset='validation', seed=42)

print("val_gen ready!")

Found 503 images belonging to 6 classes.
val_gen ready!


In [ ]:
original_model = tf.keras.models.load_model('/content/drive/MyDrive/WasteProject/best_model.keras')
new_model = tf.keras.models.load_model('/content/drive/MyDrive/WasteProject/best_model_efficientnet_gru.keras')

orig_loss, orig_acc = original_model.evaluate(val_gen, verbose=0)
new_loss, new_acc = new_model.evaluate(val_gen, verbose=0)

print("\n===== COMPARISON =====")
print(f"Original MobileNetV2:   {orig_acc*100:.2f}%")
print(f"New EfficientNetB0:     {new_acc*100:.2f}%")
print(f"Improvement:            {(new_acc - orig_acc)*100:+.2f}%")


===== COMPARISON =====
Original MobileNetV2:   28.23%
New EfficientNetB0:     86.88%
Improvement:            +58.65%


In [ ]:
# Download RealWaste dataset
os.system("kaggle datasets download -d realwaste/realwaste --unzip -p realwaste_data")
print(os.listdir("realwaste_data"))

FileNotFoundError: [Errno 2] No such file or directory: 'realwaste_data'

In [ ]:
import os
import subprocess

# Check kaggle setup
result = subprocess.run(
    ["kaggle", "datasets", "download", "-d", "realwaste/realwaste", "--unzip", "-p", "realwaste_data"],
    capture_output=True, text=True
)
print("Output:", result.stdout)
print("Error:", result.stderr)

Output: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata

Error: 


In [ ]:
os.system("kaggle datasets download -d sumn5/garbage-classification-extended --unzip -p extra_data")
import os
print(os.listdir("extra_data"))

FileNotFoundError: [Errno 2] No such file or directory: 'extra_data'

In [ ]:
import subprocess
result = subprocess.run(
    ["kaggle", "datasets", "list", "--search", "waste classification"],
    capture_output=True, text=True
)
print(result.stdout[:2000])

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication



In [ ]:
from google.colab import files
files.upload()  # select kaggle.json from your computer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
print(os.listdir('/content/drive/MyDrive/WasteProject'))

Mounted at /content/drive
['best_model.keras', 'app.py', 'requirements.txt', 'README.md', 'best_model_efficientnet_gru.keras', 'class_names.json']


In [ ]:
import os, shutil
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, models, applications
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import tensorflow as tf

# Restore combined data from Drive if exists, else recreate
if os.path.exists("/content/drive/MyDrive/WasteProject/combined_data"):
    print("Restoring from Drive...")
    shutil.copytree("/content/drive/MyDrive/WasteProject/combined_data", "combined_data", dirs_exist_ok=True)
    print("Restored!")
else:
    print("Recreating combined data...")
    os.system("kaggle datasets download -d asdasdasasdas/garbage-classification --unzip -p garbage_data")
    os.system("kaggle datasets download -d mostafaabla/garbage-classification --unzip -p extra_data")

    mapping = {'white-glass':'glass','brown-glass':'glass','green-glass':'glass',
               'paper':'paper','cardboard':'cardboard','plastic':'plastic',
               'metal':'metal','trash':'trash'}
    cats = ['glass','paper','cardboard','plastic','metal','trash']

    for cat in cats:
        os.makedirs(f"combined_data/{cat}", exist_ok=True)

    TRASHNET = "garbage_data/Garbage classification/Garbage classification/"
    EXTRA = "extra_data/garbage_classification/"

    for cat in cats:
        src = f"{TRASHNET}{cat}/"
        if os.path.exists(src):
            for img in os.listdir(src):
                shutil.copy(f"{src}{img}", f"combined_data/{cat}/{img}")

    for extra_cat, trashnet_cat in mapping.items():
        src = f"{EXTRA}{extra_cat}/"
        if os.path.exists(src):
            for img in os.listdir(src):
                shutil.copy(f"{src}{img}", f"combined_data/{trashnet_cat}/extra_{extra_cat}_{img}")
    print("Combined data ready!")

print("Images per category:")
for cat in ['glass','paper','cardboard','plastic','metal','trash']:
    print(f"  {cat}: {len(os.listdir(f'combined_data/{cat}/'))}")

Recreating combined data...
Combined data ready!
Images per category:
  glass: 2512
  paper: 1644
  cardboard: 1294
  plastic: 1347
  metal: 1179
  trash: 834


In [ ]:
import os
os.system("kaggle datasets download -d asdasdasasdas/garbage-classification --unzip -p garbage_data")
os.system("kaggle datasets download -d mostafaabla/garbage-classification --unzip -p extra_data")
print("Done!", os.listdir("extra_data/garbage_classification"))

Done! ['cardboard', 'plastic', 'battery', 'paper', 'clothes', 'shoes', 'biological', 'trash', 'metal', 'green-glass', 'white-glass', 'brown-glass']


In [ ]:
import shutil

trashnet_cats = ['glass', 'paper', 'cardboard', 'plastic', 'metal', 'trash']
mapping = {
    'white-glass':'glass','brown-glass':'glass','green-glass':'glass',
    'paper':'paper','cardboard':'cardboard','plastic':'plastic',
    'metal':'metal','trash':'trash',
}
TRASHNET = "garbage_data/Garbage classification/Garbage classification/"
EXTRA    = "extra_data/garbage_classification/"
COMBINED = "combined_data/"

for cat in trashnet_cats:
    os.makedirs(f"{COMBINED}{cat}", exist_ok=True)

for cat in trashnet_cats:
    src = f"{TRASHNET}{cat}/"
    if os.path.exists(src):
        for img in os.listdir(src):
            shutil.copy(f"{src}{img}", f"{COMBINED}{cat}/{img}")

for extra_cat, trashnet_cat in mapping.items():
    src = f"{EXTRA}{extra_cat}/"
    if os.path.exists(src):
        for img in os.listdir(src):
            shutil.copy(f"{src}{img}", f"{COMBINED}{trashnet_cat}/extra_{extra_cat}_{img}")

# Save combined_data to Drive immediately!
shutil.copytree("combined_data", "/content/drive/MyDrive/WasteProject/combined_data", dirs_exist_ok=True)

print("Combined and saved to Drive!")
for cat in trashnet_cats:
    print(f"  {cat}: {len(os.listdir(f'{COMBINED}{cat}/'))}")

Combined and saved to Drive!
  glass: 2512
  paper: 1644
  cardboard: 1294
  plastic: 1347
  metal: 1179
  trash: 834


In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, models, applications
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import tensorflow as tf, json

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True, rotation_range=20,
    zoom_range=0.2, validation_split=0.2
)
train_gen = train_datagen.flow_from_directory("combined_data/",
    target_size=(224,224), batch_size=32, class_mode='categorical',
    subset='training', seed=42)
val_gen = train_datagen.flow_from_directory("combined_data/",
    target_size=(224,224), batch_size=32, class_mode='categorical',
    subset='validation', seed=42)

base = applications.EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
base.trainable = False

inputs = layers.Input(shape=(224,224,3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(6, activation='softmax')(x)

model_combined = models.Model(inputs, outputs)
model_combined.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    ModelCheckpoint('/content/drive/MyDrive/WasteProject/best_model_combined.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Phase 1
history1 = model_combined.fit(train_gen, epochs=5,
    validation_data=val_gen, callbacks=callbacks, verbose=1)

# Phase 2
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model_combined.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy', metrics=['accuracy'])

history2 = model_combined.fit(train_gen, epochs=10,
    validation_data=val_gen, callbacks=callbacks, verbose=1)

with open('/content/drive/MyDrive/WasteProject/class_names.json', 'w') as f:
    json.dump(train_gen.class_indices, f)

all_val = history1.history['val_accuracy'] + history2.history['val_accuracy']
print(f"\nBest Val Accuracy: {max(all_val)*100:.2f}%")
print("Saved:", os.listdir('/content/drive/MyDrive/WasteProject'))

Found 7052 images belonging to 6 classes.
Found 1758 images belonging to 6 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
221/221 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7560 - loss: 0.7589
Epoch 1: val_accuracy improved from None to 0.88851, saving model to /content/drive/MyDrive/WasteProject/best_model_combined.keras

Epoch 1: finished saving model to /content/drive/MyDrive/WasteProject/best_model_combined.keras
221/221 ━━━━━━━━━━━━━━━━━━━━ 823s 4s/step - accuracy: 0.8260 - loss: 0.5383 - val_accuracy: 0.8885 - val_loss: 0.3157
Epoch 2/5
221/221 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8980 - loss: 0.2888
Epoch 2: val_accuracy improved from 0.88851 to 0.92719, saving model to /content/drive/MyDrive/WasteProject/best_model_combined.keras

Epoch 2: finished saving model to /content/drive/MyDrive/WasteProject/best_model_combined.keras
221/221 ━━━━━━━━━━━━━━━━━━━━ 800s 4s/step - accuracy: 0.9029 - loss: 0.2796 - val_accuracy: 0.9272 - val_loss: 0.2241
Ep